# 05b — Histograma de errores SIN umbral (figura para el paper)

Regenera únicamente la figura de distribución de errores de reconstrucción del autoencoder, **sin la línea de umbral**, coherente con el reporte de métricas independientes del umbral (AUC-ROC / AUC-PR).

Entrena **una sola vez** (misma semilla y configuración que el 05), así que es rápido.
Requiere el 01 (`datos_procesados/ulb.npz`). Corre todas las celdas.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")


In [ ]:
import numpy as np, time
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

gpus=tf.config.list_physical_devices("GPU")
for g in gpus: tf.config.experimental.set_memory_growth(g,True)
print(f"GPUs: {len(gpus)}" if gpus else "Sin GPU, se usara CPU")

DATOS="datos_procesados/ulb.npz"; EPOCAS=30; BATCH=256; SEMILLA=42

In [ ]:
def construir(nf):
    e=layers.Input(shape=(nf,1))
    x=layers.Conv1D(32,3,activation="relu",padding="same")(e)
    x=layers.MaxPooling1D(2,padding="same")(x)
    x=layers.Conv1D(16,3,activation="relu",padding="same")(x)
    x=layers.UpSampling1D(2)(x)
    x=layers.Conv1D(32,3,activation="relu",padding="same")(x)
    s=layers.Conv1D(1,3,activation="linear",padding="same")(x)
    m=models.Model(e,s); m.compile(optimizer="adam",loss="mse"); return m

def err_recon(m,X):
    Xr=X.reshape(-1,X.shape[1],1); rec=m.predict(Xr,batch_size=4096,verbose=0)
    L=min(Xr.shape[1],rec.shape[1]); return np.mean((Xr[:,:L,:]-rec[:,:L,:])**2,axis=(1,2))

In [ ]:
carpeta=iniciar_ejecucion("05b_histograma",{"datos":DATOS,"epocas":EPOCAS,"semilla":SEMILLA})
d=np.load(DATOS,allow_pickle=True)
Xtr,ytr=d["X_train"],d["y_train"]; Xte,yte=d["X_test"],d["y_test"]; nf=Xtr.shape[1]
Xleg=Xtr[ytr==0]

tf.keras.utils.set_random_seed(SEMILLA)
m=construir(nf)
es=tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True)
t0=time.perf_counter()
m.fit(Xleg.reshape(-1,nf,1),Xleg.reshape(-1,nf,1),epochs=EPOCAS,batch_size=BATCH,
      validation_split=0.1,callbacks=[es],verbose=2)
log(f"Entrenamiento: {time.perf_counter()-t0:.1f} s",carpeta)

et=err_recon(m,Xte)
log(f"AUC-ROC={roc_auc_score(yte,et):.4f}  AUC-PR={average_precision_score(yte,et):.4f}",carpeta)

In [ ]:
lim=np.percentile(et,99.5)
bins=np.linspace(0,lim,36)

plt.figure(figsize=(8,4.2))
plt.hist(et[yte==0],bins=bins,alpha=0.55,label="Legitimas",density=True,color="#5dade2")
plt.hist(et[yte==1],bins=bins,density=True,histtype="step",lw=2.2,color="#d35400",label="Fraude")
plt.hist(et[yte==1],bins=bins,density=True,alpha=0.18,color="#f39c12")
plt.yscale("log")
plt.xlabel("Error de reconstruccion (MSE)")
plt.ylabel("Densidad (escala log)")
plt.title("Distribucion del error de reconstruccion por clase (ULB)")
plt.legend(); plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(carpeta,"hist_errores_sin_umbral.png"),dpi=150,bbox_inches="tight")
plt.show()
log(f"\nFigura guardada en: {carpeta}",carpeta)